<a href="https://colab.research.google.com/github/varinder5680/AI/blob/main/trans_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments)
import pandas as pd


In [7]:
train_d=pd.read_csv("samsum-train.csv")
val_d=pd.read_csv("samsum-validation.csv")

In [8]:
train_d=pd.read_csv("samsum-train.csv")
val_d=pd.read_csv("samsum-validation.csv")
train_d

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14727,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14728,13828570,Theresa: <file_photo>\r\nTheresa: <file_photo>...,Theresa is at work. She gets free food and fre...
14729,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14730,13828395,Jennifer: Dear Celia! How are you doing?\r\nJe...,Celia couldn't make it to the afternoon with t...


In [9]:
train_data=train_d.sample(4000, random_state=42).reset_index(drop=True)
val_data=val_d.sample(500, random_state=42).reset_index(drop=True)
train_data

,id,dialogue,summary
0,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
1,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
2,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
3,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
4,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."
...,...,...,...
3995,13681041,Barry: hello buddy\r\nMichael: hey\r\nBarry: d...,Barry and Michael will watch football instead ...
3996,13818705,Karen: Hey Lisa. Larissa and me have recently ...,Karen and Larissa moved to Belgium and ask Lis...
3997,13821859,"Miles: Hey, guys, I'm so sorry, but I missed t...","Miles has missed the bus, so he may be 15 minu..."
3998,13812716,Emma: did you finish the book I gave you?\r\nL...,"Emma gave ""The First Fifteen Lives of Harry Au..."


In [10]:
import re
def clean_text(text):
  text=re.sub(r"\r\n", " ", text) #remove lines
  text=re.sub(r"\s+", " ", text) #remove spaces
  text=re.sub(r"<.*,!?>", " ", text) #remve html tags
  text=text.strip().lower()
  return text

In [11]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_text)
train_data["summary"]=train_data["summary"].apply(clean_text)

val_data["dialogue"]=val_d["dialogue"].apply(clean_text)
val_data["summary"]=val_d["summary"].apply(clean_text)

train_data

,id,dialogue,summary
0,13811908,violet: hi! i came across this austin's articl...,violet sent claire austin's article.
1,13716431,pat: so does anyone know when the stream is go...,pat and lou are waiting for the stream but kev...
2,13810214,jane: <gif_file> jane: whaddya think? shona: t...,jane is updating her tinder profile tonight an...
3,13729823,"adam: do u have a map of paris? tom: yes, why?...",tom has a map of paris.
4,13681400,"frank: hi, how's the family? mike: great! sam'...","mike is happy, because sam's moved out. mike a..."
...,...,...,...
3995,13681041,barry: hello buddy michael: hey barry: do you ...,barry and michael will watch football instead ...
3996,13818705,karen: hey lisa. larissa and me have recently ...,karen and larissa moved to belgium and ask lis...
3997,13821859,"miles: hey, guys, i'm so sorry, but i missed t...","miles has missed the bus, so he may be 15 minu..."
3998,13812716,emma: did you finish the book i gave you? liam...,"emma gave ""the first fifteen lives of harry au..."


In [27]:
token=T5Tokenizer.from_pretrained("t5-small")

def tokenizer_data(data):
  input=token(data["dialogue"], padding="max_length", max_length=512, truncation=True)
  target=token(data["summary"], padding="max_length", max_length=150, truncation=True)
  input["labels"]=target["input_ids"] #
  return input

In [13]:
train_dataset=train_data.apply(tokenizer_data, axis=1).tolist()
val_dataset=val_data.apply(tokenizer_data, axis=1).tolist()

In [15]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 2, 11966, 834, 9269, 3155, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

working with model



In [17]:
#NLP generation!
model=T5ForConditionalGeneration.from_pretrained("T5-small")




config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

fine tunning

In [20]:
import torch
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [21]:
train_args=TrainingArguments(
    output_dir="/.results",
    num_train_epochs=6,
    weight_decay=0.01,
    logging_steps=4,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500,



)

Trainer model

In [24]:
Trainer_modl=Trainer(
    model=model,
    args=train_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [25]:
Trainer_modl.train()

Epoch,Training Loss,Validation Loss
1,0.414359,0.375606
2,0.377292,0.355689
3,0.366095,0.350048
4,0.359122,0.346068
5,0.327499,0.344886
6,0.334246,0.344744


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9034859306812286, metrics={'train_runtime': 1291.4944, 'train_samples_per_second': 18.583, 'train_steps_per_second': 2.323, 'total_flos': 3248203235328000.0, 'train_loss': 0.9034859306812286, 'epoch': 6.0})

In [36]:

#save model

token.save_pretrained("./saved_modl")
model.save_pretrained("./saved_modl")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [37]:


token=T5Tokenizer.from_pretrained("./saved_modl")
model=T5ForConditionalGeneration.from_pretrained("./saved_modl")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [35]:
def summary_dialgue(dialogue):
  dialogue=clean_text(dialogue)
